# 04 · Does a probabilistic forecast help?

Yesterday's sales and a recent average are hard to beat in some situations. A store also needs an estimate of unusually busy days, especially while waiting for stock. I want to compare both questions using the same SKUs and dates.

The experiment chooses between two versions of LightGBM using development data. Calibration and the April assessment are separate. The notebook reads the completed run if one is available and can build it from the Part 1 artifacts otherwise.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((path for path in (cwd, *cwd.parents) if (path / "configs/config.yaml").is_file()), None)
if root is None:
    raise RuntimeError("Open the notebook from the project root or from notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_audit import load_config
from src.features import KEY, OriginStore, model_features, phase_origins

config_path = root / "configs/config.yaml"
config = load_config(config_path)
processed = root / config["data"]["processed_dir"]
focus = config["data"]["stores"][0]
if not (processed / "panel" / f"{focus}.parquet").is_file():
    raise FileNotFoundError("Run Notebook 01 before building forecast features.")

import json
from hashlib import file_digest

from src.models import load_bundle, run_forecasting, score_bundle
from src.metrics import enforce_quantiles, pinball, quantile_columns

run = root / config["forecast"]["output_dir"]
if not (run / "run_manifest.json").is_file():
    run = run_forecasting(config_path)
bundle = run / "bundle"
meta = load_bundle(bundle)
with config_path.open("rb") as file:
    config_hash = file_digest(file, "sha256").hexdigest()
if meta["config_sha256"] != config_hash:
    raise ValueError("The saved run uses a different configuration. Run the CLI in a new output directory.")
metrics = pd.read_csv(run / "metrics.csv")
display(pd.DataFrame(meta["splits"]).T)

## 1. What is a fair baseline?

| Method | What it assumes | How that assumption might fail |
|---|---|---|
| Last observation | Yesterday represents the near future | The last day may have been unusual or zero |
| Seasonal naïve | The same weekday last week repeats | Holidays, assortment changes and sparse sales can break the pattern |
| 28-day moving average | The month describes the local demand level | It smooths day-of-week patterns and responds slowly to a level shift |
| Croston | Time between sales and size on a positive day are useful separately | A stale level can linger during a long run of zeros |
| SBA | Croston's point forecast benefits from a bias adjustment | The correction does not handle every change in the underlying pattern |

The runs named Croston and SBA apply those methods to training-labeled Intermittent items and seasonal naïve to the rest. This keeps the same SKU cohort across methods. The baseline point and quantile forecasts intentionally differ. A baseline quantile uses the point forecast plus a past residual quantile for that SKU and horizon. LightGBM fits the conditional quantile directly using asymmetric pinball loss.

For LightGBM, use separate direct daily and cumulative model families, include the forecast horizon as a feature, and fit one model per requested quantile. The fit keeps every SKU and uses deterministic staggered origin sampling. An origin stride of 12 rotates through weekdays for each SKU and reduces the size of the training matrix. Scoring uses every common eligible origin. The 96 boosting rounds are a fixed compute budget, so these results do not identify the best possible tree settings.

### Try a shortcut before fitting anything

Suppose a two-day window is always either (0 units, 10 units) or (10 units, 0 units). Adding the individual-day 90th percentiles would give 20 units. The observed two-day total is always 10. The daily marginals lost information about how demand moves together, which is why we train cumulative quantiles against the sum.

In [ ]:
sample_paths = np.array([[0, 10], [10, 0]] * 10)
display(pd.DataFrame({
    "calculation": ["Sum of separate daily 90th percentiles", "90th percentile of actual two-day totals"],
    "units": [np.quantile(sample_paths, .9, axis=0).sum(), np.quantile(sample_paths.sum(axis=1), .9)],
}))

## 2. Does the as-of price feature buy us anything?

Both variants share the same calendar, demand history, boosting settings, and training origins. One adds eligible price, price age, recent price change, and the missing-price flag. Mean scaled pinball loss (SPL) across horizons and quantiles picks the variant within each family. SPL first scales each item's average loss by its training variability, then averages across items. A lower value is better. A slight advantage is enough for the predeclared choice and gives weak evidence of operational relevance.

In [ ]:
rows = [
    {"family": family, "variant": name, "development criterion": score, "selected": name == record["selected"]}
    for family, record in meta["selection"].items()
    for name, score in record["development_scores"].items()
]
display(pd.DataFrame(rows).round(5))
development = metrics.loc[
    metrics["phase"].eq("development") & metrics["regime"].eq("All") & metrics["alpha"].eq(.90)
]
display(development.pivot(index=["family", "horizon"], columns="model", values="spl").round(4))

## 3. How do we turn a prediction into a usable set of quantiles?

After choosing the price variant, the predetermined refit can use training and development labels. During calibration the fitted trees are fixed. Baseline offsets come from their observed errors per SKU and horizon. The LightGBM correction uses residuals by regime and horizon because the calibration history for one SKU is short. Consecutive windows overlap, so these empirical corrections carry no independent-sample coverage guarantee.

Separate regressions can cross. For example, the estimated 95th percentile can come out below the 90th. We preserve raw values for diagnosis, clip negative values at zero, and sort the four quantiles before using them. Calibration can reintroduce crossings, so we apply this step again. Report the strict fraction below and the inclusive fraction at or below each quantile. With integer sales and frequent zeros, the nominal level can sit between the two fractions.

## 4. What held up in April?

Report daily point errors in units and the mean item-level RMSSE, which measures squared error relative to the item's training differences. For probabilistic results use pinball, SPL, and both coverage measures. Undefined scales are counted and omitted from scaled averages. Read daily and cumulative results separately.

These are the locked forecasting results for the observed April dates. All families use the same 17 eligible origins in the default run so changes by horizon are easier to interpret. Adjacent eight-day windows overlap. Part 3 will choose policy parameters on policy-validation dates. April forecast errors cannot be used to choose those settings.

In [ ]:
test = metrics.loc[metrics["phase"].eq("final_test")]
point = test.loc[test["family"].eq("daily") & test["horizon"].eq(1) & test["alpha"].eq(.50)]
display(point[["model", "regime", "origins", "mae", "rmse", "rmsse", "rmsse_undefined_series"]]
        .sort_values(["regime", "mae"]).round(4))
cumulative = test.loc[test["family"].eq("cumulative") & test["alpha"].eq(.90)]
display(cumulative.loc[cumulative["regime"].eq("All"),
    ["model", "horizon", "origins", "pinball", "spl", "spl_undefined_series",
     "coverage_strict", "coverage_inclusive", "raw_crossing_share"]].sort_values(["horizon", "spl"]).round(4))

In [ ]:
chosen = meta["selection"]["cumulative"]["selected"]
last_horizon = max(meta["horizons"]["cumulative"])
figure, (accuracy, coverage) = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
selected_results = cumulative.loc[cumulative["horizon"].eq(last_horizon) & cumulative["regime"].ne("All")]
for label in ["moving_average", "seasonal_naive", chosen]:
    subset = selected_results.loc[selected_results["model"].eq(label)].sort_values("regime")
    accuracy.plot(subset["regime"], subset["spl"], marker="o", label=label)
accuracy.set(title=f"{last_horizon}-day 90th-percentile error", ylabel="Scaled pinball loss (lower is better)")
accuracy.legend(fontsize=8)

values = test.loc[test["family"].eq("cumulative") & test["horizon"].eq(last_horizon)
                  & test["regime"].eq("All") & test["model"].eq(chosen)].sort_values("alpha")
coverage.fill_between(values["alpha"], values["coverage_strict"], values["coverage_inclusive"],
                      alpha=.24, color="#19647E")
coverage.plot([.45, 1], [.45, 1], ":", color="#545454", label="Nominal level")
coverage.plot(values["alpha"], values["coverage_inclusive"], marker="o", color="#19647E", label="Inclusive")
coverage.plot(values["alpha"], values["coverage_strict"], marker=".", color="#19647E", linestyle="--", label="Strict")
coverage.set(title="Quantile coverage and integer ties", xlabel="Requested quantile", ylabel="Share of actuals")
coverage.legend(fontsize=8)
plt.show()

### Did the average hide differences across items?

The submitted progress report compared signed bias and paired item errors after its aggregate LightGBM result. Repeat that diagnostic using the current next-day forecasts for the same item and origin. Compute bias as prediction minus observed sales, so negative means underprediction. Use training scales to calculate the per-item RMSSE differences and keep the undefined cases visible. For the current model, the plotted point is its calibrated median. A median can be below the mean for right-skewed sales, so negative bias is a stocking concern to investigate alongside upper quantiles.

In [ ]:
selected_daily = meta["selection"]["daily"]["selected"]
observed_end = pd.Timestamp(meta["splits"]["final_test"]["end"])
last_common_origin = observed_end - pd.Timedelta(days=max(map(max, meta["horizons"].values())))
daily_points = pd.read_parquet(
    run / "forecasts/final_test_daily.parquet",
    columns=[*KEY, "model", "regime", "point", "actual"],
    filters=[("horizon", "==", 1), ("model", "in", ["moving_average", selected_daily]),
             ("origin", "<=", last_common_origin)],
)
pairs = daily_points.pivot(index=[*KEY, "regime", "actual"], columns="model", values="point").reset_index()
assert pairs[["moving_average", selected_daily]].notna().all().all()
by_item = pairs.assign(
    ma_error=pairs["moving_average"] - pairs["actual"],
    ml_error=pairs[selected_daily] - pairs["actual"],
)
by_item["ma_squared"] = by_item["ma_error"] ** 2
by_item["ml_squared"] = by_item["ml_error"] ** 2
by_item = by_item.groupby(["store_id", "item_id", "regime"], as_index=False).agg(
    ma_bias=("ma_error", "mean"), ml_bias=("ml_error", "mean"),
    ma_mse=("ma_squared", "mean"), ml_mse=("ml_squared", "mean"),
)
scale = pd.read_parquet(run / "training_scales.parquet", filters=[("family", "==", "daily")])
by_item = by_item.merge(scale[["store_id", "item_id", "sq_scale"]], validate="one_to_one")
defined = by_item["sq_scale"].where(by_item["sq_scale"] > 0)
by_item["ma_rmsse"] = np.sqrt(by_item["ma_mse"] / defined)
by_item["ml_rmsse"] = np.sqrt(by_item["ml_mse"] / defined)
by_item["paired_delta"] = by_item["ml_rmsse"] - by_item["ma_rmsse"]
by_item["ml_lower"] = by_item["paired_delta"].lt(0).where(by_item["paired_delta"].notna())
display(by_item.groupby("regime").agg(
    items=("item_id", "size"), scalable=("paired_delta", "count"), ma_bias=("ma_bias", "mean"),
    lightgbm_bias=("ml_bias", "mean"), median_rmsse_delta=("paired_delta", "median"),
    mean_rmsse_delta=("paired_delta", "mean"), lightgbm_lower_share=("ml_lower", "mean"),
).round(4))
fig, axis = plt.subplots(figsize=(8, 3.4), layout="constrained")
by_item["paired_delta"].dropna().plot.hist(bins=65, ax=axis, color="#19647E", logy=True)
axis.axvline(0, color="#b75f25", linewidth=1)
axis.set(title="Paired score differences on common April origins", xlabel="LightGBM RMSSE minus moving average RMSSE",
         ylabel="Item series (log scale)")
plt.show()

### Did calibration fix the error we care about?

Compare the same eight-day April targets before and after calibration. Both versions enforce noncrossing so the comparison isolates the addition of the residual correction. Compare pinball and coverage together. An adjustment can bring coverage closer while hurting the size-weighted penalty, or do the reverse. This is a diagnosis of the frozen choice, not an instruction to retune on April.

In [ ]:
alphas = meta["alphas"]
saved = pd.read_parquet(
    run / "forecasts/final_test_cumulative.parquet",
    filters=[("horizon", "==", last_horizon), ("model", "==", chosen)],
)
matched = saved.loc[saved["actual"].notna()]
pre = enforce_quantiles(matched[quantile_columns(alphas, "raw")])
post = matched[quantile_columns(alphas)].to_numpy()
outcomes = matched["actual"].to_numpy()
rows = []
for index, alpha in enumerate(alphas):
    for label, predictions in [("before correction", pre), ("after correction", post)]:
        predicted = predictions[:, index]
        rows.append({
            "alpha": alpha, "version": label,
            "pinball": pinball(outcomes, predicted, alpha).mean(),
            "strict": (outcomes < predicted).mean(), "inclusive": (outcomes <= predicted).mean(),
        })
display(pd.DataFrame(rows).round(4))

## 5. Can I score from the saved model near the end of the files?

Compare one set of cumulative forecasts generated from serialized models with the original run. Use the origin just before the last observed sales date. Sales beyond the file remain unknown. The saved forecast path still covers those final replenishment decisions because the calendar and origin features are available. Its eight-day target stays null.

In [ ]:
store = OriginStore.from_processed(processed, focus, config, event_codes=meta["event_codes"])
late_origin = pd.Timestamp(meta["splits"]["final_test"]["end"]) - pd.Timedelta(days=1)
reloaded = pd.concat(score_bundle(store, [late_origin], bundle), ignore_index=True)
late = saved.loc[saved["origin"].eq(late_origin) & saved["store_id"].eq(focus)].sort_values(KEY)
same = reloaded.loc[reloaded["model"].eq(chosen) & reloaded["horizon"].eq(last_horizon)].sort_values(KEY)
pd.testing.assert_frame_equal(
    late[KEY].reset_index(drop=True), same[KEY].reset_index(drop=True), check_dtype=False,
)
q = quantile_columns(alphas)
np.testing.assert_allclose(late[q], same[q], rtol=1e-6, atol=1e-6)
assert late["actual"].isna().all()
display(late[["item_id", "origin", "target_date", "actual", *q]].head(5))
print(f"PASS: serialized scoring reproduced {len(late):,} predictions at the replay tail.")

## What I would take to the simulation

In the default CA_1 run, the as-of price features reduced development SPL slightly in both model families. On the matched April origins, LightGBM did better than the moving average on next-day MAE and worse on next-day RMSSE. It also reduced eight-day 90th-percentile SPL relative to the moving average. The metric and the horizon change how we describe the result.

Hand the frozen forecasts and pre-April calibration corrections to the replenishment simulator. At equal realized service, ask whether the stronger protection-period predictor actually reduces costs. The forecasting scores cannot settle that question.